# Diagnose R-GCN And Query Projection Artifacts

Run this after training. It checks whether the exported structural models look healthy before wiring them into retrieval experiments.

In [ ]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

ARTIFACT_ROOT = ROOT / "artifacts" / "rgcn"
DATASETS = {
    "metaqa_1hop": {"structural": "metaqa_shared_graph_train", "projector": "metaqa_unified_vanilla_train"},
    "metaqa_2hop": {"structural": "metaqa_shared_graph_train", "projector": "metaqa_unified_vanilla_train"},
    "metaqa_3hop": {"structural": "metaqa_shared_graph_train", "projector": "metaqa_unified_vanilla_train"},
    "2wiki": {"structural": "2wiki_train_sample", "projector": "2wiki_train_sample"},
    "multihop_rag": {"structural": "multihop_rag_train_sample", "projector": "multihop_rag_train_sample"},
}
ARTIFACT_ROOT

## Artifact Inventory

In [ ]:
inventory = []
for name, locations in DATASETS.items():
    structural_folder = ARTIFACT_ROOT / locations["structural"]
    projector_folder = ARTIFACT_ROOT / locations["projector"]
    inventory.append(
        {
            "dataset": name,
            "folder_exists": structural_folder.exists() and projector_folder.exists(),
            "rgcn_weights": (structural_folder / "capsule_rgcn.pt").exists(),
            "rgcn_embeddings": bool(list(structural_folder.glob("capsule_rgcn_embeddings*.npz"))),
            "rgcn_metadata": (structural_folder / "capsule_rgcn_metadata.json").exists(),
            "projection_weights": (projector_folder / "query_projection_mlp.pt").exists(),
            "projection_metadata": (projector_folder / "query_projection_metadata.json").exists(),
        }
    )
inventory_df = pd.DataFrame(inventory)
inventory_df

## R-GCN Training Curves

In [ ]:
rgcn_rows = []
for name, locations in DATASETS.items():
    path = ARTIFACT_ROOT / locations["structural"] / "capsule_rgcn_metadata.json"
    if not path.exists():
        continue
    metadata = json.loads(path.read_text(encoding="utf-8"))
    for row in metadata.get("history", []):
        rgcn_rows.append({"dataset": name, **row})
rgcn_df = pd.DataFrame(rgcn_rows)
display(rgcn_df.groupby("dataset").agg(
    epochs=("epoch", "max"),
    first_val=("val_loss", "first"),
    best_val=("val_loss", "min"),
    last_val=("val_loss", "last"),
    last_train=("train_loss", "last"),
))

if not rgcn_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    for dataset, group in rgcn_df.groupby("dataset"):
        ax.plot(group["epoch"], group["val_loss"], label=f"{dataset} val")
    ax.set_title("R-GCN Validation Loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
    plt.show()

Healthy R-GCN signs:

- validation loss drops from early epochs
- validation loss does not explode while train loss falls
- artifact has many capsule vectors
- capsule vectors are not all identical

In [ ]:
def cosine_stats(vectors: np.ndarray, sample_size: int = 500) -> dict:
    if len(vectors) < 2:
        return {"capsules": len(vectors)}
    rng = np.random.default_rng(42)
    sample = vectors[rng.choice(len(vectors), size=min(sample_size, len(vectors)), replace=False)]
    sample = sample / np.clip(np.linalg.norm(sample, axis=1, keepdims=True), 1e-12, None)
    sims = sample @ sample.T
    upper = sims[np.triu_indices_from(sims, k=1)]
    return {
        "capsules": len(vectors),
        "dim": vectors.shape[1],
        "mean_pairwise_cosine": float(upper.mean()),
        "p95_pairwise_cosine": float(np.quantile(upper, 0.95)),
        "std_pairwise_cosine": float(upper.std()),
    }


vector_rows = []
for name, locations in DATASETS.items():
    folder = ARTIFACT_ROOT / locations["structural"]
    metadata_path = folder / "capsule_rgcn_metadata.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
    manifest = str(metadata.get("projection_manifest") or "capsule_rgcn_embeddings.npz")
    path = folder / manifest.split(":", 1)[0]
    if not path.exists():
        vector_rows.append({"dataset": name, "missing": True})
        continue
    payload = np.load(path, allow_pickle=True)
    stats = cosine_stats(payload["embeddings"].astype("float32"))
    vector_rows.append({"dataset": name, "missing": False, **stats})
vector_df = pd.DataFrame(vector_rows)
vector_df

## Projection Training Curves

In [ ]:
projection_rows = []
projection_summary = []
for name, locations in DATASETS.items():
    path = ARTIFACT_ROOT / locations["projector"] / "query_projection_metadata.json"
    if not path.exists():
        projection_summary.append({"dataset": name, "missing": True})
        continue
    metadata = json.loads(path.read_text(encoding="utf-8"))
    history = metadata.get("history", [])
    projection_summary.append(
        {
            "dataset": name,
            "missing": False,
            "triplets": metadata.get("triplets"),
            "input_dim": metadata.get("input_dim"),
            "output_dim": metadata.get("output_dim"),
            "best_val_loss": metadata.get("best_val_loss"),
            "epochs": len(history),
            "last_val_pos": history[-1].get("val_pos") if history else None,
            "last_val_neg": history[-1].get("val_neg") if history else None,
            "last_margin": (history[-1].get("val_pos") - history[-1].get("val_neg")) if history else None,
        }
    )
    for row in history:
        projection_rows.append({"dataset": name, **row})

projection_summary_df = pd.DataFrame(projection_summary)
display(projection_summary_df)

projection_df = pd.DataFrame(projection_rows)
if not projection_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for dataset, group in projection_df.groupby("dataset"):
        axes[0].plot(group["epoch"], group["val_loss"], label=dataset)
        axes[1].plot(group["epoch"], group["val_pos"] - group["val_neg"], label=dataset)
    axes[0].set_title("Projection Validation Loss")
    axes[1].set_title("Projection Validation Margin")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend()
    plt.show()

Projection warnings to look for:

- missing projection artifacts
- very few triplets
- validation loss exactly `0.0` for many epochs, which can mean negatives are too easy
- `val_pos - val_neg` not clearly positive